# Deep Dive: Comprehensive Audio Analysis of a Single Song

Before trusting any cross-song similarity pipeline, let's get a solid ground-truth
understanding of what's actually going on inside one song's audio. This notebook
computes and visualizes every major representation and feature type, on one track
you choose.

**What's covered:**
1. Waveform
2. STFT spectrogram (linear frequency, Fourier-transform based)
3. Mel spectrogram (perceptually-weighted frequency)
4. Constant-Q Transform (log-frequency, note-aligned)
5. Chromagram (harmony/key content, octave-collapsed)
6. Tempogram (rhythm over time)
7. MFCCs (compact timbre summary)
8. Harmonic/percussive source separation (HPSS)
9. Self-similarity matrix (structure: verses, choruses, repeats)
10. Scalar features over time (spectral centroid, contrast, rolloff, zero-crossing rate)
11. Tempo/beat tracking
12. Tonnetz (tonal centroid / harmonic relationships)

In [ ]:
# --- Setup ---
!pip install -q librosa soundfile matplotlib scipy plotly

import numpy as np
import librosa
import librosa.display
import matplotlib.pyplot as plt
from IPython.display import Audio, display, HTML
import base64
import io

print('Setup done.')

In [ ]:
# --- Upload a single song ---
from google.colab import files
import os

os.makedirs('analysis_audio', exist_ok=True)
uploaded = files.upload()

song_path = None
for fname in uploaded.keys():
    dest = os.path.join('analysis_audio', fname)
    os.rename(fname, dest)
    song_path = dest  # uses the last uploaded file if multiple

print('Analyzing:', song_path)

In [ ]:
# --- Load audio ---
SR = 22050  # standard analysis sample rate (good balance of detail vs speed)

y, sr = librosa.load(song_path, sr=SR, mono=True)
duration = len(y) / sr
print(f'Loaded {duration:.1f}s of audio at {sr}Hz ({len(y)} samples)')
display(Audio(y, rate=sr))

## 1. Waveform — what the song looks like as pure loudness over time

This is the simplest possible picture of a sound: at every tiny instant, how loud is
it? That's all a waveform shows. The wiggly line going up and down is loudness rising
and falling — a tall spike means a loud moment, a flat stretch means quiet or silence.

It doesn't tell you anything about *pitch* or *tone* yet — just "loud" vs "quiet," from
instant to instant.

**Why we start here:** every other analysis in this notebook — and the actual
recommendation system we're building toward — is computed FROM this raw signal. Before
we can ask a computer to compare "does this song sound like that one," we first need to
look at what a song even *is* to a computer: just a very long list of loudness numbers,
changing many thousands of times per second. Nothing here is the final answer — it's
the ground floor everything else gets built on, so it's worth actually seeing it once
before we start transforming it into fancier things.

**Below is interactive:** press play on the audio, and watch the red line sweep across
the plot in sync with what you're hearing. You can also click anywhere on the plot to
jump playback straight to that moment — useful for lining up "what does *that* spike
sound like?"

(The technical name for this view is a *waveform* — you may see that word elsewhere.)

In [ ]:
import plotly.graph_objects as go

def audio_to_base64_wav(y_audio, sr_audio):
    import soundfile as sf
    buf = io.BytesIO()
    sf.write(buf, y_audio, sr_audio, format='WAV')
    buf.seek(0)
    return base64.b64encode(buf.read()).decode()

def synced_line_plot(x_vals, y_vals, sr_audio, y_audio, title, x_label='Time (seconds)', y_label='Loudness', div_id='waveform-plot', audio_id='waveform-audio'):
    """A plotly line plot with a red playhead that syncs to an audio player, and click-to-seek."""
    duration_local = len(y_audio) / sr_audio

    fig = go.Figure()
    fig.add_trace(go.Scatter(x=x_vals, y=y_vals, mode='lines', line=dict(width=1, color='#1f77b4'), name=y_label))
    fig.add_shape(type='line', x0=0, x1=0, y0=min(y_vals), y1=max(y_vals),
                  line=dict(color='red', width=2), name='playhead')
    fig.update_layout(height=320, margin=dict(l=50, r=20, t=40, b=40),
                       xaxis_title=x_label, yaxis_title=y_label, title=title,
                       showlegend=False)

    plot_html = fig.to_html(include_plotlyjs='cdn', full_html=False, div_id=div_id)
    b64_audio = audio_to_base64_wav(y_audio, sr_audio)

    html = f"""
    {plot_html}
    <audio id="{audio_id}" controls style="width:100%; margin-top:10px;">
      <source src="data:audio/wav;base64,{b64_audio}" type="audio/wav">
    </audio>
    <p style="font-size:13px; color:gray;">Tip: click anywhere on the plot to jump playback to that moment.</p>
    <script>
    (function() {{
        var audio = document.getElementById('{audio_id}');
        var plotDiv = document.getElementById('{div_id}');

        audio.addEventListener('timeupdate', function() {{
            var x = audio.currentTime;
            Plotly.relayout(plotDiv, {{'shapes[0].x0': x, 'shapes[0].x1': x}});
        }});

        plotDiv.on('plotly_click', function(data) {{
            var x = data.points[0].x;
            audio.currentTime = x;
            audio.play();
        }});
    }})();
    </script>
    """
    display(HTML(html))

# Downsample for smooth browser rendering -- plotting every single sample would be too slow/heavy
downsample_factor = max(1, len(y) // 4000)
y_ds = y[::downsample_factor]
t_ds = np.linspace(0, duration, len(y_ds))

synced_line_plot(t_ds, y_ds, sr, y, title='Waveform -- click to jump, red line follows playback',
                  y_label='Loudness (amplitude)', div_id='waveform-plot', audio_id='waveform-audio')

## 2. Spectrogram — loudness AND pitch over time

The waveform above only told you "loud or quiet." This next picture adds the missing
piece: *which pitches* are present at each moment, and how strong each one is.

Think of it like a piano roll crossed with a heat map: time runs left to right (same
as before), but now the vertical axis is frequency — low pitches near the bottom, high
pitches near the top. Brighter/warmer color at a point means that pitch is louder at
that instant. A held note shows up as a bright horizontal band; a drum hit shows up as
a bright vertical smear across many frequencies at once (since noise-like sounds spread
energy across the whole range).

**Why this is the next step:** the waveform alone is too crude to compare songs by — two
completely different-sounding songs can have similar loudness patterns, and two songs
that sound alike to your ear can have very different loudness patterns. What actually
makes something sound like a specific instrument, a specific texture, a specific "tickle"
is almost entirely about which pitches/frequencies are present and how they interact —
none of which the waveform shows. The spectrogram is the first representation in this
notebook that's actually rich enough to be useful for similarity comparison later on, and
it's also the direct input CLAP (the AI model we use for matching) itself consumes
internally, in a slightly reshaped form (the mel spectrogram, coming up next).

**How this is actually computed:** we can't get "which pitches are present" from a
single instant alone — pitch only makes sense over a little stretch of time. So the
computer looks at a small sliding window of the waveform (a few milliseconds at a time),
and for each window asks "how much of this window looks like a slow wave vs a fast
wave vs a really fast wave" — slow waves are low pitches, fast waves are high pitches.
That question-per-window, repeated across the whole song, is what builds this picture.
The formal name for this technique is the *Short-Time Fourier Transform (STFT)* — it's
the same Fourier transform from signal processing, just applied to short windows
instead of the whole song at once, so we keep the timing information instead of losing it.

**Same interactivity as before:** cyan line follows playback, click to jump.

In [ ]:
def synced_heatmap_plot(times, freqs, values, sr_audio, y_audio, title,
                          x_label='Time (seconds)', y_label='Frequency (Hz)',
                          colorbar_label='dB', div_id='heatmap-plot', audio_id='heatmap-audio',
                          log_y=False):
    """Same idea as synced_line_plot, but for 2D time-frequency images (spectrograms etc)."""
    fig = go.Figure()
    fig.add_trace(go.Heatmap(x=times, y=freqs, z=values, colorscale='Magma',
                              colorbar=dict(title=colorbar_label)))
    fig.add_shape(type='line', x0=0, x1=0, y0=min(freqs), y1=max(freqs),
                  line=dict(color='cyan', width=2), name='playhead')
    fig.update_layout(height=380, margin=dict(l=60, r=20, t=40, b=40),
                       xaxis_title=x_label, yaxis_title=y_label, title=title)
    if log_y:
        fig.update_yaxes(type='log')

    plot_html = fig.to_html(include_plotlyjs='cdn', full_html=False, div_id=div_id)
    b64_audio = audio_to_base64_wav(y_audio, sr_audio)

    html = f"""
    {plot_html}
    <audio id="{audio_id}" controls style="width:100%; margin-top:10px;">
      <source src="data:audio/wav;base64,{b64_audio}" type="audio/wav">
    </audio>
    <p style="font-size:13px; color:gray;">Tip: click anywhere on the plot to jump playback to that moment.</p>
    <script>
    (function() {{
        var audio = document.getElementById('{audio_id}');
        var plotDiv = document.getElementById('{div_id}');

        audio.addEventListener('timeupdate', function() {{
            var x = audio.currentTime;
            Plotly.relayout(plotDiv, {{'shapes[0].x0': x, 'shapes[0].x1': x}});
        }});

        plotDiv.on('plotly_click', function(data) {{
            var x = data.points[0].x;
            audio.currentTime = x;
            audio.play();
        }});
    }})();
    </script>
    """
    display(HTML(html))


D = librosa.stft(y)
S_db = librosa.amplitude_to_db(np.abs(D), ref=np.max)

freqs_hz = librosa.fft_frequencies(sr=sr)
times_stft = librosa.frames_to_time(np.arange(S_db.shape[1]), sr=sr)

synced_heatmap_plot(times_stft, freqs_hz, S_db, sr, y,
                     title='Spectrogram -- brighter = louder at that pitch and moment',
                     y_label='Frequency (Hz)', colorbar_label='dB',
                     div_id='stft-plot', audio_id='stft-audio')

## 3. Mel spectrogram

Same STFT data, but the frequency axis is reweighted to match human pitch perception —
we're far more sensitive to differences at low frequencies than high ones. This is what
CLAP and most audio ML models actually consume internally.

In [ ]:
mel = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128)
mel_db = librosa.power_to_db(mel, ref=np.max)

plt.figure(figsize=(12, 4))
librosa.display.specshow(mel_db, sr=sr, x_axis='time', y_axis='mel')
plt.colorbar(format='%+2.0f dB')
plt.title('Mel spectrogram')
plt.tight_layout()
plt.show()

## 4. Constant-Q Transform (CQT)

Like the STFT, but frequency bins are spaced logarithmically — matching musical
octaves/notes instead of equal Hz steps. Much better for capturing pitch/harmonic
content precisely, which plain STFT smears at low frequencies.

In [ ]:
C = librosa.cqt(y=y, sr=sr)
C_db = librosa.amplitude_to_db(np.abs(C), ref=np.max)

plt.figure(figsize=(12, 4))
librosa.display.specshow(C_db, sr=sr, x_axis='time', y_axis='cqt_note')
plt.colorbar(format='%+2.0f dB')
plt.title('Constant-Q Transform')
plt.tight_layout()
plt.show()

## 5. Chromagram

Collapses all octaves of the same pitch class together (every C, regardless of octave,
becomes one bin). Captures harmony and key, strips out octave and timbre — useful for
comparing chord progressions/tonal character across songs independent of instrumentation.

In [ ]:
chroma = librosa.feature.chroma_cqt(y=y, sr=sr)

plt.figure(figsize=(12, 4))
librosa.display.specshow(chroma, sr=sr, x_axis='time', y_axis='chroma')
plt.colorbar()
plt.title('Chromagram (CQT-based)')
plt.tight_layout()
plt.show()

## 6. Tempogram

Captures rhythmic/tempo patterns over time — how strong periodic pulses are at
different tempos (BPM), and how that changes through the song (e.g. a build-up
increasing in intensity, a tempo shift into a bridge).

In [ ]:
onset_env = librosa.onset.onset_strength(y=y, sr=sr)
tempogram = librosa.feature.tempogram(onset_envelope=onset_env, sr=sr)

plt.figure(figsize=(12, 4))
librosa.display.specshow(tempogram, sr=sr, x_axis='time', y_axis='tempo', cmap='magma')
plt.colorbar()
plt.title('Tempogram')
plt.tight_layout()
plt.show()

## 7. MFCCs (Mel-Frequency Cepstral Coefficients)

A compressed summary of the spectral envelope — historically the workhorse timbre
feature for speech/music. Much lower-dimensional than a full spectrogram, so it's
compact and easy to compare directly, at the cost of losing fine detail.

In [ ]:
mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=20)

plt.figure(figsize=(12, 4))
librosa.display.specshow(mfcc, sr=sr, x_axis='time')
plt.colorbar()
plt.title('MFCCs')
plt.ylabel('MFCC coefficient index')
plt.tight_layout()
plt.show()

## 8. Harmonic/percussive source separation (HPSS)

Splits the audio into its harmonic component (pitched, sustained sounds — melody,
chords) and percussive component (transient, noisy sounds — drums, attacks). Each can
be analyzed or compared separately — useful if "the tickle" lives more in one than the other.

In [ ]:
y_harmonic, y_percussive = librosa.effects.hpss(y)

print('Harmonic component:')
display(Audio(y_harmonic, rate=sr))
print('Percussive component:')
display(Audio(y_percussive, rate=sr))

fig, axes = plt.subplots(2, 1, figsize=(12, 5), sharex=True)
librosa.display.waveshow(y_harmonic, sr=sr, ax=axes[0])
axes[0].set_title('Harmonic component')
librosa.display.waveshow(y_percussive, sr=sr, ax=axes[1])
axes[1].set_title('Percussive component')
plt.tight_layout()
plt.show()

## 9. Self-similarity matrix (structure)

Compares the song against itself over time using a feature representation (chroma
here, since harmony tends to repeat with song structure). Bright diagonal stripes
parallel to the main diagonal reveal repeated sections — verse/chorus structure —
without needing labeled data.

In [ ]:
chroma_sync = librosa.util.sync(chroma, librosa.beat.beat_track(y=y, sr=sr)[1])
R = librosa.segment.recurrence_matrix(chroma_sync, mode='affinity', sym=True)

plt.figure(figsize=(7, 7))
librosa.display.specshow(R, x_axis='time', y_axis='time', sr=sr)
plt.title('Self-similarity matrix (beat-synced chroma)')
plt.tight_layout()
plt.show()

print('Look for bright parallel stripes off the main diagonal — those mark repeated sections.')

## 10. Scalar features over time

Single-number-per-frame descriptors, each capturing a different quality:
- **Spectral centroid**: "brightness" — higher = more high-frequency energy
- **Spectral contrast**: peak-vs-valley strength across frequency bands — texture/timbre richness
- **Spectral rolloff**: frequency below which most energy is concentrated
- **Zero-crossing rate**: how often the signal crosses zero — rough proxy for noisiness/percussiveness

In [ ]:
centroid = librosa.feature.spectral_centroid(y=y, sr=sr)[0]
contrast = librosa.feature.spectral_contrast(y=y, sr=sr)
rolloff = librosa.feature.spectral_rolloff(y=y, sr=sr)[0]
zcr = librosa.feature.zero_crossing_rate(y)[0]

times = librosa.frames_to_time(np.arange(len(centroid)), sr=sr)

fig, axes = plt.subplots(3, 1, figsize=(12, 7), sharex=True)

axes[0].plot(times, centroid, linewidth=0.8, label='Spectral centroid')
axes[0].plot(times, rolloff, linewidth=0.8, label='Spectral rolloff', alpha=0.7)
axes[0].legend(loc='upper right', fontsize=8)
axes[0].set_ylabel('Hz')
axes[0].set_title('Brightness-related features')

librosa.display.specshow(contrast, x_axis='time', sr=sr, ax=axes[1])
axes[1].set_title('Spectral contrast (per frequency band)')

axes[2].plot(times, zcr, linewidth=0.8, color='darkred')
axes[2].set_title('Zero-crossing rate')
axes[2].set_xlabel('Time (s)')

plt.tight_layout()
plt.show()

print(f'Mean spectral centroid: {centroid.mean():.1f} Hz')
print(f'Mean spectral rolloff:  {rolloff.mean():.1f} Hz')
print(f'Mean zero-crossing rate: {zcr.mean():.4f}')

## 11. Tempo / beat tracking

Estimates the overall tempo (BPM) and the precise timestamps of detected beats.

In [ ]:
tempo, beat_frames = librosa.beat.beat_track(y=y, sr=sr)
beat_times = librosa.frames_to_time(beat_frames, sr=sr)

print(f'Estimated tempo: {float(tempo):.1f} BPM')
print(f'Number of detected beats: {len(beat_times)}')

plt.figure(figsize=(12, 3))
librosa.display.waveshow(y, sr=sr, alpha=0.6)
plt.vlines(beat_times, ymin=-1, ymax=1, color='red', linewidth=0.6, alpha=0.7, label='detected beats')
plt.legend()
plt.title(f'Beat tracking (tempo ~ {float(tempo):.1f} BPM)')
plt.tight_layout()
plt.show()

## 12. Tonnetz (tonal centroid features)

Projects harmonic content into a 6-dimensional space representing tonal relationships
(fifths, minor/major thirds) — captures harmonic character in a very compact form,
often used alongside chroma for key/harmony-based comparison.

In [ ]:
tonnetz = librosa.feature.tonnetz(y=y_harmonic, sr=sr)

plt.figure(figsize=(12, 4))
librosa.display.specshow(tonnetz, sr=sr, x_axis='time', cmap='coolwarm')
plt.colorbar()
plt.title('Tonnetz')
plt.ylabel('Tonal dimension')
plt.tight_layout()
plt.show()

## 13. Implicit representation: CLAP embeddings over time

Everything above (MFCCs, chroma, spectral contrast, tonnetz) is an **explicit**
feature — a human designed the equation, so it can only capture qualities someone
already thought to name. Below, we slide a window across the same song and embed
each window with CLAP — an **implicit** representation, learned from data rather
than hand-designed. It may capture qualities that don't have a name at all.

In [ ]:
# --- Load CLAP (via transformers) ---
!pip install -q --upgrade transformers accelerate

import torch
from transformers import ClapModel, ClapProcessor

device = 'cuda' if torch.cuda.is_available() else 'cpu'
clap_model = ClapModel.from_pretrained("laion/clap-htsat-unfused").to(device)
clap_processor = ClapProcessor.from_pretrained("laion/clap-htsat-unfused")
clap_model.eval()
print('CLAP loaded on', device)

In [ ]:
# --- Slide a window across the song and embed each with CLAP ---

CLAP_SR = 48000
WIN_SEC = 5.0
HOP_SEC = 1.0   # smaller hop than the cross-song POC, since we want a fine-grained timeline for ONE song

y_clap, _ = librosa.load(song_path, sr=CLAP_SR, mono=True)

win_len = int(WIN_SEC * CLAP_SR)
hop_len = int(HOP_SEC * CLAP_SR)

windows = []
window_times = []
for start in range(0, max(len(y_clap) - win_len, 1), hop_len):
    windows.append(y_clap[start:start + win_len])
    window_times.append(start / CLAP_SR)

def extract_tensor(output):
    if isinstance(output, torch.Tensor):
        return output
    if hasattr(output, 'pooler_output') and output.pooler_output is not None:
        return output.pooler_output
    if hasattr(output, 'last_hidden_state'):
        return output.last_hidden_state.mean(dim=1)
    return output[0]

clap_embeddings = []
batch_size = 8
with torch.no_grad():
    for i in range(0, len(windows), batch_size):
        batch = [np.asarray(w, dtype=np.float32) for w in windows[i:i+batch_size]]
        inputs = clap_processor(audio=batch, sampling_rate=CLAP_SR, return_tensors='pt').to(device)
        raw = clap_model.get_audio_features(**inputs)
        emb = extract_tensor(raw)
        clap_embeddings.append(emb.cpu().numpy())

clap_embeddings = np.concatenate(clap_embeddings, axis=0)
print(f'Embedded {len(windows)} windows -> shape {clap_embeddings.shape}')

## 14. Self-similarity: explicit (chroma) vs implicit (CLAP) side by side

Same idea as section 9, computed two ways. If both agree on where the repeated
sections are, that's a good sign both representations are picking up real structure.
Where they disagree is genuinely interesting — it means CLAP is responding to
something chroma (harmony) doesn't capture, like texture or production character.

In [ ]:
def cosine_sim_matrix(mat):
    norm = mat / (np.linalg.norm(mat, axis=1, keepdims=True) + 1e-8)
    return norm @ norm.T

clap_sim = cosine_sim_matrix(clap_embeddings)

fig, axes = plt.subplots(1, 2, figsize=(13, 6))

librosa.display.specshow(R, x_axis='time', y_axis='time', sr=sr, ax=axes[0])
axes[0].set_title('Explicit: chroma self-similarity')

im = axes[1].imshow(clap_sim, cmap='magma', origin='lower',
                     extent=[window_times[0], window_times[-1], window_times[0], window_times[-1]])
axes[1].set_title('Implicit: CLAP self-similarity')
axes[1].set_xlabel('Time (s)')
fig.colorbar(im, ax=axes[1])

plt.tight_layout()
plt.show()

## 15. Embedding trajectory: PCA vs UMAP

Both squash the high-dimensional CLAP embedding down to 2D so we can look at it, but
differently. PCA finds the directions of greatest linear variance — fast, deterministic,
but can flatten meaningful non-linear structure. UMAP preserves local neighborhoods —
points that are genuinely close in the real embedding space tend to stay close in 2D —
which usually reveals cluster shape more clearly, at the cost of being non-deterministic
and a bit slower.

Color = time progression through the song, so you can see the actual path the song
traces through embedding space as it plays.

In [ ]:
!pip install -q umap-learn

from sklearn.decomposition import PCA
import umap

pca_coords = PCA(n_components=2).fit_transform(clap_embeddings)
umap_coords = umap.UMAP(n_neighbors=10, min_dist=0.1, random_state=42).fit_transform(clap_embeddings)

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

sc0 = axes[0].scatter(pca_coords[:, 0], pca_coords[:, 1], c=window_times, cmap='viridis', s=25)
axes[0].plot(pca_coords[:, 0], pca_coords[:, 1], alpha=0.25, linewidth=0.8, color='gray')
axes[0].set_title('PCA projection (colored by time)')
fig.colorbar(sc0, ax=axes[0], label='time (s)')

sc1 = axes[1].scatter(umap_coords[:, 0], umap_coords[:, 1], c=window_times, cmap='viridis', s=25)
axes[1].plot(umap_coords[:, 0], umap_coords[:, 1], alpha=0.25, linewidth=0.8, color='gray')
axes[1].set_title('UMAP projection (colored by time)')
fig.colorbar(sc1, ax=axes[1], label='time (s)')

plt.tight_layout()
plt.show()

## 16. Summarizing the song's template (DBSCAN clustering)

This is the actual "template" answer: cluster the time windows by embedding
similarity, without pre-specifying how many sections exist. DBSCAN groups windows
into a handful of recurring "states" (verse-like, chorus-like, bridge-like) based on
density in embedding space, and labels anything that doesn't fit a cluster as noise
(-1) — useful for catching one-off transitions or intros.

Reading the cluster labels in time order gives you a compressed structural sequence —
e.g. `A-A-B-A-A-B-C-B` — which IS the song's form, discovered rather than hand-labeled.

In [ ]:
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler

# DBSCAN is distance-based, so standardize first
scaled = StandardScaler().fit_transform(clap_embeddings)

# eps is the trickiest DBSCAN parameter -- start here and adjust if everything
# ends up as one cluster (lower eps) or all noise (raise eps)
dbscan = DBSCAN(eps=3.0, min_samples=3)
cluster_labels = dbscan.fit_predict(scaled)

n_clusters = len(set(cluster_labels)) - (1 if -1 in cluster_labels else 0)
n_noise = list(cluster_labels).count(-1)
print(f'Found {n_clusters} structural clusters, {n_noise} windows marked as noise/transition')

# Template as a readable letter sequence (A, B, C... ; noise shown as ".")
label_map = {}
next_letter = ord('A')
template_str = ''
for lab in cluster_labels:
    if lab == -1:
        template_str += '.'
        continue
    if lab not in label_map:
        label_map[lab] = chr(next_letter)
        next_letter += 1
    template_str += label_map[lab]

print('\nSong template (structural sequence):')
print(template_str)

In [ ]:
# --- Visualize the template as a colored timeline strip ---

unique_clusters = sorted(set(cluster_labels))
cmap = plt.get_cmap('tab10')
color_map = {lab: ('lightgray' if lab == -1 else cmap(i % 10)) for i, lab in enumerate(unique_clusters)}

fig, ax = plt.subplots(figsize=(12, 1.5))
for i, (t, lab) in enumerate(zip(window_times, cluster_labels)):
    width = HOP_SEC
    ax.barh(0, width, left=t, color=color_map[lab], edgecolor='none')

ax.set_yticks([])
ax.set_xlabel('Time (s)')
ax.set_title('Structural template over time (each color = one recurring state)')
plt.tight_layout()
plt.show()

print('If eps=3.0 gave you one giant cluster or all noise, adjust eps in the cell above and re-run --')
print('lower eps = stricter/more clusters, higher eps = looser/fewer clusters.')

## 17. Train your own autoencoder (a second implicit representation)

CLAP is implicit but pretrained on millions of tracks it never saw here. An
autoencoder we train ourselves, directly on this song's own windows, is implicit
in a different way — it learns whatever patterns are needed to reconstruct *this*
specific audio, nothing more. It won't generalize like CLAP does, but comparing the
two tells us something real: does a representation tuned to just this song organize
structure differently than one trained broadly on the world's music?

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

# --- Compute mel-spectrograms for the same windows used for CLAP (section 13) ---
N_MELS = 64
mel_specs = []
for w in windows:
    m = librosa.feature.melspectrogram(y=np.asarray(w, dtype=np.float32), sr=CLAP_SR, n_mels=N_MELS)
    m_db = librosa.power_to_db(m, ref=np.max)
    mel_specs.append(m_db)

mel_specs = np.stack(mel_specs)  # (N, n_mels, time_frames)
mel_min, mel_max = mel_specs.min(), mel_specs.max()
mel_norm = (mel_specs - mel_min) / (mel_max - mel_min + 1e-8)
print('Mel-spectrogram batch shape:', mel_norm.shape)

In [ ]:
# --- A small convolutional autoencoder ---

class ConvAutoencoder(nn.Module):
    def __init__(self, latent_dim=32):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 16, 3, stride=2, padding=1), nn.ReLU(),
            nn.Conv2d(16, 32, 3, stride=2, padding=1), nn.ReLU(),
            nn.Conv2d(32, 64, 3, stride=2, padding=1), nn.ReLU(),
            nn.AdaptiveAvgPool2d((4, 4)),
        )
        self.fc_enc = nn.Linear(64 * 4 * 4, latent_dim)
        self.fc_dec = nn.Linear(latent_dim, 64 * 4 * 4)
        self.decoder = nn.Sequential(
            nn.Upsample(scale_factor=2), nn.Conv2d(64, 32, 3, padding=1), nn.ReLU(),
            nn.Upsample(scale_factor=2), nn.Conv2d(32, 16, 3, padding=1), nn.ReLU(),
            nn.Upsample(scale_factor=2), nn.Conv2d(16, 1, 3, padding=1), nn.Sigmoid(),
        )

    def encode(self, x):
        h = self.encoder(x)
        return self.fc_enc(h.view(h.size(0), -1))

    def decode(self, z, target_shape):
        h = self.fc_dec(z).view(z.size(0), 64, 4, 4)
        out = self.decoder(h)
        return F.interpolate(out, size=target_shape, mode='bilinear', align_corners=False)

    def forward(self, x):
        z = self.encode(x)
        return self.decode(z, x.shape[-2:]), z

ae = ConvAutoencoder(latent_dim=32).to(device)
optimizer = torch.optim.Adam(ae.parameters(), lr=1e-3)

x_train = torch.tensor(mel_norm, dtype=torch.float32).unsqueeze(1).to(device)  # (N, 1, n_mels, time)
print('Training tensor shape:', x_train.shape)

In [ ]:
# --- Train (quick -- a minute or two on GPU) ---

EPOCHS = 40
BATCH = 16

n = x_train.shape[0]
losses = []

for epoch in range(EPOCHS):
    perm = torch.randperm(n)
    epoch_loss = 0.0
    for i in range(0, n, BATCH):
        idx = perm[i:i+BATCH]
        batch = x_train[idx]
        recon, z = ae(batch)
        loss = F.mse_loss(recon, batch)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * batch.size(0)
    epoch_loss /= n
    losses.append(epoch_loss)
    if epoch % 10 == 0 or epoch == EPOCHS - 1:
        print(f'Epoch {epoch:3d}  loss={epoch_loss:.5f}')

plt.figure(figsize=(8, 3))
plt.plot(losses)
plt.title('Autoencoder training loss')
plt.xlabel('epoch')
plt.ylabel('MSE loss')
plt.tight_layout()
plt.show()

In [ ]:
# --- Extract autoencoder embeddings for every window, and peek at a reconstruction ---

ae.eval()
with torch.no_grad():
    ae_embeddings = ae.encode(x_train).cpu().numpy()

print('Autoencoder embedding shape:', ae_embeddings.shape)

# Sanity check: does it actually reconstruct? Compare one window's real vs reconstructed spectrogram
check_idx = 0
with torch.no_grad():
    recon, _ = ae(x_train[check_idx:check_idx+1])
recon_np = recon.squeeze().cpu().numpy()

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].imshow(mel_norm[check_idx], origin='lower', aspect='auto', cmap='magma')
axes[0].set_title('Original mel-spectrogram')
axes[1].imshow(recon_np, origin='lower', aspect='auto', cmap='magma')
axes[1].set_title('Autoencoder reconstruction')
plt.tight_layout()
plt.show()

print('Listen to the original window this reconstruction is based on:')
display(Audio(windows[check_idx], rate=CLAP_SR))

## 18. Three-way self-similarity: explicit vs two implicit representations

Chroma (explicit, hand-designed) vs CLAP (implicit, broadly pretrained) vs your own
autoencoder (implicit, trained only on this song). Where all three agree, that's
robust structure. Where they diverge, each is picking up on something different.

In [ ]:
ae_sim = cosine_sim_matrix(ae_embeddings)

fig, axes = plt.subplots(1, 3, figsize=(16, 5.5))

librosa.display.specshow(R, x_axis='time', y_axis='time', sr=sr, ax=axes[0])
axes[0].set_title('Explicit: chroma')

extent = [window_times[0], window_times[-1], window_times[0], window_times[-1]]
im1 = axes[1].imshow(clap_sim, cmap='magma', origin='lower', extent=extent)
axes[1].set_title('Implicit: CLAP (pretrained)')

im2 = axes[2].imshow(ae_sim, cmap='magma', origin='lower', extent=extent)
axes[2].set_title('Implicit: autoencoder (self-trained)')

for ax in axes[1:]:
    ax.set_xlabel('Time (s)')

plt.tight_layout()
plt.show()

## 19. Combining representations

You don't have to pick one. Z-score normalizing each representation (so no single
one dominates just because its numbers are larger) and concatenating them gives you
one combined vector per window that carries information from all sources at once.

In [ ]:
from sklearn.preprocessing import StandardScaler

clap_z = StandardScaler().fit_transform(clap_embeddings)
ae_z = StandardScaler().fit_transform(ae_embeddings)

combined_embeddings = np.concatenate([clap_z, ae_z], axis=1)
combined_sim = cosine_sim_matrix(combined_embeddings)

print('Combined embedding shape:', combined_embeddings.shape, '(CLAP dims + autoencoder dims)')

plt.figure(figsize=(6.5, 6))
plt.imshow(combined_sim, cmap='magma', origin='lower', extent=extent)
plt.title('Combined (CLAP + autoencoder) self-similarity')
plt.xlabel('Time (s)')
plt.colorbar()
plt.tight_layout()
plt.show()

## 20. Listen to each structural cluster

The DBSCAN template from section 16 assigned every window to a cluster letter. Here's
one representative audio example from each cluster, so you can actually hear what the
algorithm grouped together — the real test of whether the clustering means anything.

In [ ]:
seen = {}
for i, lab in enumerate(cluster_labels):
    if lab not in seen:
        seen[lab] = i

for lab in sorted(seen.keys()):
    idx = seen[lab]
    name = 'noise/transition' if lab == -1 else f'cluster {label_map[lab]}'
    print(f'{name}  (first seen at {window_times[idx]:.1f}s):')
    display(Audio(windows[idx], rate=CLAP_SR))

## Summary: all features in one place

A single feature vector per song, combining everything above into fixed-length
summary statistics (mean + std per feature) — this is the kind of vector you'd feed
into the interpretability/re-ranking layer of the real project, or use as a simpler
hand-crafted baseline to compare against CLAP's deep embeddings.

In [ ]:
def summarize(feat, name):
    return {f'{name}_mean': np.mean(feat), f'{name}_std': np.std(feat)}

summary = {}
summary.update(summarize(centroid, 'spectral_centroid'))
summary.update(summarize(rolloff, 'spectral_rolloff'))
summary.update(summarize(zcr, 'zero_crossing_rate'))
summary.update(summarize(contrast, 'spectral_contrast'))
summary.update(summarize(mfcc, 'mfcc'))
summary.update(summarize(chroma, 'chroma'))
summary.update(summarize(tonnetz, 'tonnetz'))
summary['tempo_bpm'] = float(tempo)

import pandas as pd
display(pd.DataFrame([summary]).T.rename(columns={0: 'value'}))